# NB 09 — SHAP Explainability (Stage C, side-by-side)

TreeSHAP on the tuned CatBoost + LinearSHAP on the tuned LogReg pipeline. Each gets a bar + beeswarm plot. 3 waterfall plots from the deployed model (or dominant base model if ensemble adopted).

In [1]:
import os, sys, time, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT))
else:
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, joblib, shap
from catboost import CatBoostClassifier

from src.config import DATA_PROCESSED, MODELS, REPORTS, SEED
from src.utils import seed_all
from src.explain.shap_utils import transform_for_shap, get_explainer, compute_shap_values, top_k_features

seed_all(SEED)
FIG_DIR = REPORTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)


ROOT: D:\AgrNav


In [2]:
deployed = json.loads((MODELS / "deployed.json").read_text(encoding="utf-8"))
print("deployed:", json.dumps(deployed, indent=2, default=str))
dominant = deployed["dominant_model"]


deployed: {
  "deployed_name": "catboost",
  "deployed_family": "catboost",
  "dominant_model": "catboost",
  "dominant_family": "catboost",
  "adopt_ensemble": false,
  "ensemble_winner_strategy": "rank_average",
  "ensemble_winner_weights": {
    "catboost": 0.3333333333333333,
    "lightgbm": 0.3333333333333333,
    "logreg": 0.3333333333333333
  },
  "ensemble_test_auc": 0.7846050293781279,
  "best_single_name": "catboost",
  "best_single_test_auc": 0.7869
}


In [3]:
df = pd.read_parquet(DATA_PROCESSED / "df_master_extended.parquet")
idx = np.load(DATA_PROCESSED / "split_indices.npz")
train_idx, test_idx = idx["train_idx"], idx["test_idx"]

train_sorted = df.iloc[train_idx].sort_values("visit_date")
drop = ["visit_date", "sale_within_7d"]
X_train = train_sorted.drop(columns=drop).reset_index(drop=True)
X_test = df.iloc[test_idx].drop(columns=drop).reset_index(drop=True)
y_test = df.iloc[test_idx]["sale_within_7d"].astype(int).reset_index(drop=True)
print("X_train", X_train.shape, "X_test", X_test.shape)


X_train (23862, 28) X_test (6138, 28)


In [4]:
# Load CatBoost
cb_model = CatBoostClassifier()
cb_model.load_model(str(MODELS / "catboost_optuna_best.cbm"))
print("CatBoost loaded")

# Load LR Pipeline
lr_pipe = joblib.load(MODELS / "logreg_optuna_best.pkl")
print("LR pipeline loaded:", lr_pipe)


CatBoost loaded
LR pipeline loaded: Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('sentinel_skew',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 missing_values=-1,
                                                                                 strategy='median')),
                                                                  ('log1p',
                                                                   FunctionTransformer(feature_names_out='one-to-one',
                                                                                       func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                       

In [5]:
# === CatBoost branch ===
t0 = time.perf_counter()
X_cb_test, cb_feat_names = transform_for_shap(cb_model, X_test, "catboost")
cb_explainer = shap.TreeExplainer(cb_model)
cb_shap = compute_shap_values(cb_explainer, X_cb_test)
print(f"CatBoost SHAP shape={cb_shap.shape} computed in {time.perf_counter()-t0:.1f}s")

# Save values
cb_shap_df = pd.DataFrame(cb_shap, columns=cb_feat_names)
cb_shap_df.insert(0, "test_index", test_idx)
cb_shap_df.to_parquet(DATA_PROCESSED / "shap_values_catboost.parquet", index=False)

cb_top10 = top_k_features(cb_shap, cb_feat_names, k=10)
print("CatBoost top-10:")
print(cb_top10.to_string(index=False))


CatBoost SHAP shape=(6138, 28) computed in 0.3s
CatBoost top-10:
 rank                        feature  mean_abs_shap  mean_signed_shap
    1            terr_prod_conv_rate       0.639097          0.033758
    2      days_since_last_sale_terr       0.350092          0.054749
    3              sku_qty_pre_visit       0.275062         -0.179997
    4                 sales_velocity       0.149627         -0.000619
    5       rolling_avg_weekly_sales       0.136877          0.024162
    6 days_since_product_last_pushed       0.105681         -0.008223
    7                         rep_id       0.082654          0.001621
    8        stock_to_velocity_weeks       0.057988          0.001806
    9               stock_change_wow       0.055982          0.012274
   10                 week_of_season       0.042455         -0.036427


In [6]:
# Plots — CatBoost
plt.figure()
shap.summary_plot(cb_shap, X_cb_test, plot_type="bar", max_display=10, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_top10_bar_catboost.png", dpi=120, bbox_inches="tight")
plt.close()

plt.figure()
shap.summary_plot(cb_shap, X_cb_test, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_beeswarm_catboost.png", dpi=120, bbox_inches="tight")
plt.close()
print("saved CatBoost SHAP figures")


saved CatBoost SHAP figures


In [7]:
# === LogReg branch ===
bg_size = 2000
X_lr_bg = X_train.sample(bg_size, random_state=SEED)
X_lr_bg_t, lr_feat_names = transform_for_shap(lr_pipe, X_lr_bg, "lr_nb_knn")
X_lr_test_t, _ = transform_for_shap(lr_pipe, X_test, "lr_nb_knn")

# Convert to dense for LinearExplainer (the OHE+scaled output may be sparse).
import scipy.sparse as sp
def _to_dense(M):
    return M.toarray() if sp.issparse(M) else M

X_lr_bg_arr = _to_dense(X_lr_bg_t.to_numpy() if isinstance(X_lr_bg_t, pd.DataFrame) else X_lr_bg_t)
X_lr_test_arr = _to_dense(X_lr_test_t.to_numpy() if isinstance(X_lr_test_t, pd.DataFrame) else X_lr_test_t)

t0 = time.perf_counter()
lr_explainer = shap.LinearExplainer(lr_pipe.named_steps["est"], X_lr_bg_arr)
lr_shap = lr_explainer.shap_values(X_lr_test_arr)
if isinstance(lr_shap, list):
    lr_shap = np.asarray(lr_shap[1] if len(lr_shap) == 2 else lr_shap[0])
print(f"LR SHAP shape={lr_shap.shape} computed in {time.perf_counter()-t0:.1f}s")

lr_shap_df = pd.DataFrame(lr_shap, columns=lr_feat_names)
lr_shap_df.insert(0, "test_index", test_idx)
lr_shap_df.to_parquet(DATA_PROCESSED / "shap_values_logreg.parquet", index=False)

lr_top10 = top_k_features(lr_shap, lr_feat_names, k=10)
print("LR top-10:")
print(lr_top10.to_string(index=False))


LR SHAP shape=(6138, 80) computed in 0.0s
LR top-10:
 rank                        feature  mean_abs_shap  mean_signed_shap
    1            terr_prod_conv_rate       0.248978          0.091771
    2        stock_to_velocity_weeks       0.128446          0.052193
    3      days_since_last_sale_terr       0.118133          0.038497
    4                 sales_velocity       0.110790          0.007703
    5              sku_qty_pre_visit       0.077154         -0.041029
    6                 week_of_season       0.041982         -0.041982
    7          product_sold_last_14d       0.030587          0.008230
    8          rep_product_diversity       0.012534         -0.012328
    9 days_since_product_last_pushed       0.011730         -0.006101
   10               stock_change_wow       0.010430          0.005406


In [8]:
# Plots — LR
plt.figure()
shap.summary_plot(lr_shap, X_lr_test_arr, plot_type="bar", feature_names=lr_feat_names,
                  max_display=10, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_top10_bar_logreg.png", dpi=120, bbox_inches="tight")
plt.close()

plt.figure()
shap.summary_plot(lr_shap, X_lr_test_arr, feature_names=lr_feat_names,
                  max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_beeswarm_logreg.png", dpi=120, bbox_inches="tight")
plt.close()
print("saved LR SHAP figures")


saved LR SHAP figures


In [9]:
# Waterfalls using deployed (or dominant if ensemble) model.
test_preds_path = DATA_PROCESSED / f"phase2_test_preds_{dominant}.parquet"
test_preds = pd.read_parquet(test_preds_path).set_index("test_index")
top3_idx = test_preds["y_proba"].nlargest(3).index.tolist()
print("top-3 test rows by probability:", top3_idx)

if dominant == "catboost":
    explainer_used = cb_explainer
    X_used = X_cb_test
    base_val = float(np.atleast_1d(cb_explainer.expected_value).ravel()[-1])
    shap_used = cb_shap
    feat_names_used = cb_feat_names
elif dominant == "logreg":
    explainer_used = lr_explainer
    X_used = pd.DataFrame(X_lr_test_arr, columns=lr_feat_names)
    base_val = float(np.atleast_1d(lr_explainer.expected_value).ravel()[-1])
    shap_used = lr_shap
    feat_names_used = lr_feat_names
else:
    # lightgbm dominant — fall back to CatBoost waterfalls (we didn't load LGBM SHAP)
    print(f"[note] dominant={dominant}; using CatBoost SHAP for waterfalls instead")
    explainer_used = cb_explainer
    X_used = X_cb_test
    base_val = float(np.atleast_1d(cb_explainer.expected_value).ravel()[-1])
    shap_used = cb_shap
    feat_names_used = cb_feat_names

# Positional indices in test_idx
pos_lookup = {idx: i for i, idx in enumerate(test_idx)}
for k, idx_val in enumerate(top3_idx, 1):
    pos = pos_lookup[idx_val]
    plt.figure()
    expl_obj = shap.Explanation(
        values=shap_used[pos],
        base_values=base_val,
        data=(X_used.iloc[pos].to_numpy() if hasattr(X_used, "iloc") else X_used[pos]),
        feature_names=feat_names_used,
    )
    shap.plots.waterfall(expl_obj, max_display=12, show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"shap_waterfall_{k}.png", dpi=120, bbox_inches="tight")
    plt.close()
print("saved 3 waterfall plots")


top-3 test rows by probability: [5976, 1077, 21139]


saved 3 waterfall plots


In [10]:
# Hard-stop: CatBoost top-3 must overlap >=2 with Phase 1 importance top-5
P1_TOP5 = {"terr_prod_conv_rate", "days_since_last_sale_terr", "sku_qty_pre_visit",
           "days_since_product_last_pushed", "rolling_avg_weekly_sales"}

cb_top3 = set(cb_top10["feature"].head(3))
overlap_cb = cb_top3 & P1_TOP5
lr_top3 = set(lr_top10["feature"].head(3))
overlap_lr = lr_top3 & P1_TOP5
print(f"CatBoost top-3: {cb_top3} -> overlap {overlap_cb}")
print(f"LR top-3: {lr_top3} -> overlap {overlap_lr}")
assert len(overlap_cb) >= 2, (
    f"HARD STOP [SHAP]: CatBoost top-3 overlap with Phase 1 top-5 = {overlap_cb} (<2)"
)
print("hard-stop check passed (CatBoost overlap >= 2)")


CatBoost top-3: {'days_since_last_sale_terr', 'sku_qty_pre_visit', 'terr_prod_conv_rate'} -> overlap {'days_since_last_sale_terr', 'sku_qty_pre_visit', 'terr_prod_conv_rate'}
LR top-3: {'days_since_last_sale_terr', 'stock_to_velocity_weeks', 'terr_prod_conv_rate'} -> overlap {'days_since_last_sale_terr', 'terr_prod_conv_rate'}
hard-stop check passed (CatBoost overlap >= 2)


In [11]:
# shap_top10.md — side-by-side
narrative = []
narrative.append(f"Deployed model: **{deployed['deployed_name']}** "
                 f"({'ensemble' if deployed['adopt_ensemble'] else 'single'}; "
                 f"dominant base = `{dominant}`).")
narrative.append(f"CatBoost top-3 ↔ Phase 1 importance top-5 overlap: {sorted(overlap_cb)} "
                 f"({len(overlap_cb)}/3).")
narrative.append(f"LR top-3 ↔ Phase 1 importance top-5 overlap: {sorted(overlap_lr)} "
                 f"({len(overlap_lr)}/3). LR explains via signed linear coefficients on a "
                 f"scaled/one-hot-encoded feature space, so its top features differ structurally "
                 f"from tree-importance rankings.")

lines = [
    "# SHAP Top-10 (Phase 2 Stage C)",
    "",
    *narrative,
    "",
    "## CatBoost (TreeExplainer)",
    "",
    cb_top10.to_markdown(index=False, floatfmt=".4f"),
    "",
    "## LogReg (LinearExplainer)",
    "",
    lr_top10.to_markdown(index=False, floatfmt=".4f"),
    "",
    "## Figures",
    "",
    "- `reports/figures/shap_top10_bar_catboost.png`",
    "- `reports/figures/shap_top10_bar_logreg.png`",
    "- `reports/figures/shap_beeswarm_catboost.png`",
    "- `reports/figures/shap_beeswarm_logreg.png`",
    "- `reports/figures/shap_waterfall_{1,2,3}.png`  (from dominant model = `" + dominant + "`)",
    "",
]
(REPORTS / "shap_top10.md").write_text("\n".join(lines), encoding="utf-8")
print("wrote reports/shap_top10.md")


wrote reports/shap_top10.md
